In [ ]:
#!/usr/bin/env python3
# compare_models_from_dirs_with_errorbars.py
# Read ALL *test_results*.csv files under each model directory (recursively),
# aggregate to 1 score per subject, compute metrics with bootstrap CIs,
# and make scatter plots WITH per-subject error bars (SD or SEM).

from pathlib import Path
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

'''
python compare_models_from_dirs_with_errorbars.py \
  --outdir "Compare_Model" \
  --agg mean \
  --err sd      # or: --err sem

'''

# --------- EDIT THESE DIRECTORIES/LABELS IF NEEDED ----------
MODEL_DIRS = {
    "FA": "NEW COLUMNS GNC/FA/column_md_results/",
    "MD": "NEW COLUMNS GNC/MD/column_md_results/",
    "Thickness": "NEW COLUMNS GNC/Thickness/thickness_model_results/",
    "MD+Thickness": "NEW COLUMNS GNC/MD+thickness/column_md_results/",
    "QSM": "NEW COLUMNS GNC/QSM/column_md_results/",
    "QSM+MD+Thickness": "NEW COLUMNS GNC/MD+thickness+QSM_dual_channel/column_md_results/",
    "MD+Thickness+QSM+PCs": "NEW COLUMNS GNC/MD+thickness+QSM+PCs/column_md_results/",
}

TEST_RESULTS_PATTERN = "**/*test_results*_ref.csv"

SUBJECT_CANDIDATES = [
    "subject_id","subject","sub_id","participant_id","participant",
    "rid","RID","id","ID","ptid","PTID"
]

# --------- helpers ---------

# colors
BLUE = "#0B3D91"   # deep blue
RED  = "red"       # dashed identity line

def make_scatter_with_errbars(model_name, tab, outdir, m, err_mode, agg):
    """Pred vs True (subject-level) with vertical error bars (SD or SEM)."""
    x = tab["real_age_years"].values
    y = tab["predicted_age"].values
    yerr = _yerr_from_sd_sem(tab["pred_sd"].values, tab["n_preds"].values, err_mode)

    plt.figure()
    # deep blue dots + error bars
    plt.errorbar(
        x, y, yerr=yerr, fmt='o', capsize=3,
        color=BLUE, ecolor=BLUE, markerfacecolor=BLUE, markeredgecolor=BLUE,
        alpha=0.95, elinewidth=1
    )

    lo = min(x.min(), y.min()); hi = max(x.max(), y.max())
    # dashed red identity line
    plt.plot([lo, hi], [lo, hi], linestyle='--', color=RED, linewidth=1.6)

    # regression Pred ~ True (leave default color)
    xc = x - x.mean(); denom = np.sum(xc*xc)
    if denom > 0:
        beta = np.sum(xc*(y - y.mean()))/denom
        alpha = y.mean() - beta * x.mean()
        xs = np.linspace(x.min(), x.max(), 100); ys = alpha + beta*xs
        plt.plot(xs, ys)

    plt.xlabel("True age (years)")
    plt.ylabel(f"Predicted age (years) — {agg} ± {err_mode.upper()}")
    plt.title(f"{model_name}: Predicted vs True (subject-level {agg}, {err_mode.upper()} bars)")
    txt = (
        f"MAE:  {m['MAE']:.2f}  [95% CI {m['MAE_CI95_lo']:.2f}, {m['MAE_CI95_hi']:.2f}] yrs\n"
        f"RMSE: {m['RMSE']:.2f} [95% CI {m['RMSE_CI95_lo']:.2f}, {m['RMSE_CI95_hi']:.2f}] yrs\n"
        f"R²:   {m['R2']:.3f} [95% CI {m['R2_CI95_lo']:.3f}, {m['R2_CI95_hi']:.3f}]\n"
        f"β (Pred~True): {m['beta_pred_true']:.3f} "
        f"[{m['beta_pred_true_CI95_lo']:.3f}, {m['beta_pred_true_CI95_hi']:.3f}]\n"
        f"α: {m['alpha_pred_true']:.2f}"
    )
    ax = plt.gca()
    ax.text(0.98, 0.03, txt, transform=ax.transAxes, ha="right", va="bottom",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85), fontsize=9)
    plt.tight_layout()
    plt.savefig(outdir / f"{model_name}_pred_vs_true_subject_{agg}_{err_mode}.png", dpi=200)
    plt.close()


def make_bag_plot_with_errbars(model_name, tab, outdir, m, err_mode, agg):
    """BAG vs Age (subject-level) with vertical error bars (SD or SEM)."""
    x = tab["real_age_years"].values
    y = tab["BAG"].values
    yerr = _yerr_from_sd_sem(tab["bag_sd"].values, tab["n_preds"].values, err_mode)

    plt.figure()
    # deep blue dots + error bars
    plt.errorbar(
        x, y, yerr=yerr, fmt='o', capsize=3,
        color=BLUE, ecolor=BLUE, markerfacecolor=BLUE, markeredgecolor=BLUE,
        alpha=0.95, elinewidth=1
    )

    # regression BAG ~ Age (default color)
    xc = x - x.mean(); denom = np.sum(xc*xc)
    if denom > 0:
        beta = np.sum(xc*(y - y.mean()))/denom
        alpha = y.mean() - beta * x.mean()
        xs = np.linspace(x.min(), x.max(), 100); ys = alpha + beta*xs
        plt.plot(xs, ys)

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xlabel("True age (years)")
    plt.ylabel(f"BAG (years) — {agg} ± {err_mode.upper()}")
    plt.title(f"{model_name}: BAG vs Age (subject-level {agg}, {err_mode.upper()} bars)")
    txt = (
        f"β (BAG~Age): {m['beta_bag_age']:.3f} "
        f"[95% CI {m['beta_bag_age_CI95_lo']:.3f}, {m['beta_bag_age_CI95_hi']:.3f}]"
    )
    ax = plt.gca()
    ax.text(0.98, 0.03, txt, transform=ax.transAxes, ha="right", va="bottom",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85), fontsize=9)
    plt.tight_layout()
    plt.savefig(outdir / f"{model_name}_bag_vs_age_subject_{agg}_{err_mode}.png", dpi=200)
    plt.close()


def detect_cols(df):
    cols = {c.lower(): c for c in df.columns}
    y_true = cols.get("y_true") or cols.get("age") or cols.get("true_age")
    y_pred = cols.get("y_pred") or cols.get("yhat") or cols.get("y_hat") \
             or cols.get("pred") or cols.get("pred_age") or cols.get("predicted_age")
    if y_true is None or y_pred is None:
        raise ValueError(f"Missing y_true/y_pred in columns: {list(df.columns)}")
    subj = None
    for cand in SUBJECT_CANDIDATES:
        if cand in df.columns:
            subj = cand
            break
    # return original-cased column names
    return df.columns[df.columns.str.lower()==y_true][0], \
           df.columns[df.columns.str.lower()==y_pred][0], \
           subj

def mae(y_true, y_pred):  return float(np.mean(np.abs(y_pred - y_true)))
def rmse(y_true, y_pred): return float(np.sqrt(np.mean((y_pred - y_true)**2)))
def r2_score(y_true, y_pred):
    ss_res = float(np.sum((y_true - y_pred)**2))
    ss_tot = float(np.sum((y_true - np.mean(y_true))**2))
    return 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan

def slope_and_intercept(x, y):
    x = np.asarray(x); y = np.asarray(y)
    xc = x - x.mean(); denom = float(np.sum(xc*xc))
    if denom == 0: return np.nan, np.nan
    beta = float(np.sum(xc*(y - y.mean())) / denom)
    alpha = float(y.mean() - beta * x.mean())
    return beta, alpha

def bootstrap_subject_cis(y_true, y_pred, n_boot=10000, seed=123):
    """Bootstrap 95% CIs by resampling subjects (indices)."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    MAEs  = np.empty(n_boot); RMSEs = np.empty(n_boot); R2s = np.empty(n_boot)
    Bpred = np.empty(n_boot);  Alph = np.empty(n_boot);  Bbag = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]; yp = y_pred[idx]; bag = yp - yt
        MAEs[b]  = mae(yt, yp)
        RMSEs[b] = rmse(yt, yp)
        R2s[b]   = r2_score(yt, yp)
        bp, ap   = slope_and_intercept(yt, yp)
        bb, _    = slope_and_intercept(yt, bag)
        Bpred[b] = bp; Alph[b] = ap; Bbag[b] = bb
    pct = lambda arr: (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))
    return {
        "MAE_CI":  pct(MAEs),
        "RMSE_CI": pct(RMSEs),
        "R2_CI":   pct(R2s),
        "beta_pred_true_CI": pct(Bpred),
        "alpha_pred_true_CI": pct(Alph),
        "beta_bag_age_CI": pct(Bbag),
    }

def collect_test_results(model_dir: Path):
    files = sorted([p for p in model_dir.glob(TEST_RESULTS_PATTERN)
                    if p.is_file() and p.suffix.lower()==".csv"])
    return files

def per_model_subject_table(all_rows: pd.DataFrame, agg="mean"):
    """Aggregate to one row per subject (mean or median) and compute SD/SEM."""
    # compute bag per row before aggregation
    all_rows = all_rows.copy()
    all_rows["bag"] = all_rows["y_pred"] - all_rows["y_true"]

    # aggregation
    g = all_rows.groupby("subject_id", as_index=False).agg(
        real_age_years=("y_true", agg),
        predicted_age=("y_pred", agg),
        n_preds=("y_pred","size"),
        pred_sd=("y_pred","std"),
        bag_mean=("bag", agg),
        bag_sd=("bag","std"),
    )
    g["pred_sd"] = g["pred_sd"].fillna(0.0)
    g["bag_sd"]  = g["bag_sd"].fillna(0.0)
    # For clarity, also keep BAG = predicted - true from the aggregated means
    g["BAG"] = g["predicted_age"] - g["real_age_years"]
    return g

def _yerr_from_sd_sem(sd, n, mode):
    if mode == "sem":
        return np.where(n > 1, sd / np.sqrt(n), 0.0)
    return sd  # "sd"



def make_comparison_bars(summary_df, outdir):
    models = summary_df["model"].tolist()
    x = np.arange(len(models))
    # MAE
    plt.figure()
    plt.bar(x, summary_df["MAE"],
            yerr=[summary_df["MAE"]-summary_df["MAE_CI95_lo"],
                  summary_df["MAE_CI95_hi"]-summary_df["MAE"]],
            capsize=4)
    plt.xticks(x, models, rotation=60)
    plt.ylabel("MAE (years)")
    plt.title("Model comparison: MAE (95% CI)")
    plt.tight_layout()
    plt.savefig(outdir / "compare_MAE.png", dpi=200)
    plt.close()
    # RMSE
    plt.figure()
    plt.bar(x, summary_df["RMSE"],
            yerr=[summary_df["RMSE"]-summary_df["RMSE_CI95_lo"],
                  summary_df["RMSE_CI95_hi"]-summary_df["RMSE"]],
            capsize=4)
    plt.xticks(x, models, rotation=60)
    plt.ylabel("RMSE (years)")
    plt.title("Model comparison: RMSE (95% CI)")
    plt.tight_layout()
    plt.savefig(outdir / "compare_RMSE.png", dpi=200)
    plt.close()
    # R²
    plt.figure()
    plt.bar(x, summary_df["R2"])
    plt.xticks(x, models, rotation=60)
    plt.ylabel("R²")
    plt.title("Model comparison: R²")
    plt.tight_layout()
    plt.savefig(outdir / "compare_R2.png", dpi=200)
    plt.close()

# --------- main ---------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--outdir", default="Compare_Model",
                    help="Output directory for plots/CSVs")
    ap.add_argument("--agg", choices=["mean","median"], default="mean",
                    help="Aggregate per subject (mean default; median robust)")
    ap.add_argument("--err", choices=["sd","sem"], default="sd",
                    help="Error bars on scatter: SD (default) or SEM")
    ap.add_argument("--boots", type=int, default=10000, help="Bootstrap iterations (subjects)")
    ap.add_argument("--seed", type=int, default=123, help="Bootstrap seed")
    # In notebooks, IPython/Jupyter injects kernel args (like --f=...);
    # use parse_known_args() so unknown kernel arguments are ignored when running inside a notebook
    args, _ = ap.parse_known_args()

    outdir = Path(args.outdir).expanduser(); outdir.mkdir(parents=True, exist_ok=True)

    summaries = []
    for model, dir_path in MODEL_DIRS.items():
        model_dir = Path(dir_path).expanduser()
        files = collect_test_results(model_dir)
        if not files:
            print(f"[WARN] No *test_results*_ref.csv under {model_dir} for {model}")
            continue
        print(f"{model}: found {len(files)} test_results file(s).")

        frames = []
        for f in files:
            df = pd.read_csv(f)
            ytc, ypc, subj = detect_cols(df)
            take = df[[ytc, ypc]].rename(columns={ytc:"y_true", ypc:"y_pred"}).copy()
            if subj is not None:
                take["subject_id"] = df[subj].astype(str).values
            else:
                take["subject_id"] = np.arange(take.shape[0]).astype(str)
            frames.append(take)

        all_rows = pd.concat(frames, ignore_index=True)
        tab = per_model_subject_table(all_rows, agg=args.agg)

        yt = tab["real_age_years"].values
        yp = tab["predicted_age"].values
        bag = tab["BAG"].values

        # point estimates
        MAE = mae(yt, yp); RMSE = rmse(yt, yp); R2 = r2_score(yt, yp)
        beta_pred, alpha_pred = slope_and_intercept(yt, yp)
        beta_bag, _ = slope_and_intercept(yt, bag)

        # bootstrap CIs over subjects
        cis = bootstrap_subject_cis(yt, yp, n_boot=args.boots, seed=args.seed)

        m = {
            "model": model, "N_subjects": len(tab),
            "MAE": MAE, "MAE_CI95_lo": cis["MAE_CI"][0], "MAE_CI95_hi": cis["MAE_CI"][1],
            "RMSE": RMSE, "RMSE_CI95_lo": cis["RMSE_CI"][0], "RMSE_CI95_hi": cis["RMSE_CI"][1],
            "R2": R2, "R2_CI95_lo": cis["R2_CI"][0], "R2_CI95_hi": cis["R2_CI"][1],
            "beta_pred_true": beta_pred,
            "beta_pred_true_CI95_lo": cis["beta_pred_true_CI"][0],
            "beta_pred_true_CI95_hi": cis["beta_pred_true_CI"][1],
            "alpha_pred_true": alpha_pred,
            "alpha_pred_true_CI95_lo": cis["alpha_pred_true_CI"][0],
            "alpha_pred_true_CI95_hi": cis["alpha_pred_true_CI"][1],
            "beta_bag_age": beta_bag,
            "beta_bag_age_CI95_lo": cis["beta_bag_age_CI"][0],
            "beta_bag_age_CI95_hi": cis["beta_bag_age_CI"][1],
        }
        summaries.append(m)

        # Save per-model subject table & plots (with error bars)
        tab.to_csv(outdir / f"{model}_subject_level_{args.agg}.csv", index=False)
        make_scatter_with_errbars(model, tab, outdir, m, args.err, args.agg)
        make_bag_plot_with_errbars(model, tab, outdir, m, args.err, args.agg)

    if not summaries:
        print("No models processed. Check directories and filenames.")
        return

    summary_df = pd.DataFrame(summaries).sort_values("model")
    summary_df.to_csv(outdir / f"model_metrics_subject_{args.agg}.csv", index=False)
    print("\n=== Subject-level metrics (per model, ALL *test_results*.csv) ===")
    print(summary_df[["model","N_subjects","MAE","MAE_CI95_lo","MAE_CI95_hi","RMSE","RMSE_CI95_lo","RMSE_CI95_hi","R2","beta_pred_true","beta_bag_age"]].to_string(index=False))

    make_comparison_bars(summary_df, outdir)
    print(f"\nSaved plots and CSVs to: {outdir}")

if __name__ == "__main__":
    main()

FA: found 70 test_results file(s).
MD: found 70 test_results file(s).
MD: found 70 test_results file(s).
Thickness: found 70 test_results file(s).
Thickness: found 70 test_results file(s).
MD+Thickness: found 70 test_results file(s).
MD+Thickness: found 70 test_results file(s).
QSM: found 70 test_results file(s).
QSM: found 70 test_results file(s).
QSM+MD+Thickness: found 70 test_results file(s).
QSM+MD+Thickness: found 70 test_results file(s).
MD+Thickness+QSM+PCs: found 70 test_results file(s).
MD+Thickness+QSM+PCs: found 70 test_results file(s).

=== Subject-level metrics (per model, ALL *test_results*.csv) ===
               model  N_subjects      MAE  MAE_CI95_lo  MAE_CI95_hi      RMSE  RMSE_CI95_lo  RMSE_CI95_hi       R2  beta_pred_true  beta_bag_age
                  FA          59 6.826327     5.511830     8.250479  8.720562      7.009857     10.530937 0.623062        0.582877     -0.417123
                  MD          59 5.879150     4.611286     7.238417  7.844952      6.364

: 